# Routing by Model Tier

The Routing pattern usually sends a request to a specialised *handler* — story, poem, joke.
This notebook routes on a different axis: it sends the request to a specialised **model
tier**.

A cheap model looks at each request and decides whether it is hard. Easy work goes to the
cheap model; hard work goes to a reasoning model. You pay reasoning prices only on the
traffic that needs them.

This is the applied counterpart to two Foundations notebooks. Those measured the tradeoff
on fixed tasks. This one makes the choice **per request, at runtime**, and measures whether
the routing actually paid.

## Learning objectives

1. Build a LangGraph workflow whose conditional edge selects a model tier, not a handler.
2. Measure the router against both baselines — all-cheap and all-reasoning — on one workload.
3. Reason about the router's own cost, since triage is itself a model call.
4. Choose a fallback direction deliberately, and say why one direction is safe and the other is not.

## Where this fits

- `01_Foundations/00_Theory_and_Foundations/Reasoning_and_Model_Selection/` — the two
  notebooks behind this one: *which model*, and *how much effort*.
- Sibling: `Routing.ipynb` in this folder routes to handlers. Same pattern, different axis.

## Prerequisites and cost

`OPENAI_API_KEY` in the project-root `.env`. Roughly `3 × len(WORKLOAD)` calls plus triage —
about 30 by default, a few cents.

## 1. Two constraints worth knowing before the setup cell

**This phase mandates the `helpers` factory.** `02_Core/05_AI_Agent_Fundamentals/CLAUDE.md`
says to use `get_llm()` for LangGraph notebooks, so that is what we use — unlike the two
Foundations notebooks, which are forbidden from using it.

Two things will bite you if you do not know them:

1. **`reasoning_effort` only reaches providers that can express it.** `get_llm()` raises
   rather than silently ignoring the argument, because a dropped parameter would make you
   conclude "effort made no difference" from a measurement that never happened. We pin
   `provider="openai"` here so the two tiers differ only in model, not in provider.
2. **o-series models reject a custom temperature.** `get_llm()` defaults to `temperature=0`,
   which is right for the cheap tier and an error for the reasoning tier. We pass
   `temperature=1` there. This is the kind of thing that fails on your first real call, not
   at construction time.

In [ ]:
# ============ SETUP ============
import time
from typing_extensions import Literal, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from helpers import get_llm

load_dotenv()

# Cheap tier: answers directly, also does the triage.
cheap_llm = get_llm(provider="openai", model="gpt-4o-mini", temperature=0)

# Reasoning tier. temperature=1 because o-series rejects anything else;
# reasoning_effort reaches the model via the passthrough added to get_llm().
reasoning_llm = get_llm(
    provider="openai",
    model="o4-mini",
    temperature=1,
    reasoning_effort="high",
)

## 2. The triage schema

The router has exactly one job: decide `easy` or `hard`. Keeping the decision to a
two-value enum matters — a router that can emit free text is a router that can emit
something your graph has no edge for.

`with_structured_output` makes the model return the enum rather than a sentence about the
enum.

In [ ]:
# ============ TRIAGE SCHEMA ============
class Triage(BaseModel):
    """How hard is this request?"""

    difficulty: Literal["easy", "hard"] = Field(
        description=(
            "'hard' if answering needs multi-step reasoning, constraint satisfaction, "
            "or arithmetic across several dependent steps. "
            "'easy' if it is lookup, extraction, formatting, or a single obvious step."
        )
    )


triage_router = cheap_llm.with_structured_output(Triage)

In [ ]:
# ============ GRAPH STATE ============
class TierState(TypedDict):
    question: str        # the incoming request
    difficulty: str      # what triage decided
    answer: str          # final answer
    tier_used: str       # which model actually answered — for the cost accounting
    seconds: float       # wall-clock for the answering call only

## 3. The nodes

Three nodes: one triages, two answer. The answering nodes are deliberately identical except
for the model they call — that is the whole experiment. If they differed in prompt as well,
any measured difference would be unattributable.

In [ ]:
# ============ NODES ============
def triage(state: TierState) -> dict:
    """Cheap model decides whether this needs the expensive tier."""
    decision = triage_router.invoke([
        SystemMessage(content="Classify how hard this request is to answer correctly."),
        HumanMessage(content=state["question"]),
    ])
    return {"difficulty": decision.difficulty}


def _answer_with(llm, state: TierState, tier: str) -> dict:
    started = time.perf_counter()
    reply = llm.invoke([HumanMessage(content=state["question"])])
    return {
        "answer": (reply.content or "").strip(),
        "tier_used": tier,
        "seconds": time.perf_counter() - started,
    }


def answer_cheap(state: TierState) -> dict:
    return _answer_with(cheap_llm, state, "cheap")


def answer_reasoning(state: TierState) -> dict:
    return _answer_with(reasoning_llm, state, "reasoning")


def pick_tier(state: TierState) -> Literal["answer_cheap", "answer_reasoning"]:
    """The conditional edge. Selects a model tier, not a handler."""
    return "answer_reasoning" if state["difficulty"] == "hard" else "answer_cheap"

In [ ]:
# ============ BUILD THE GRAPH ============
builder = StateGraph(TierState)
builder.add_node("triage", triage)
builder.add_node("answer_cheap", answer_cheap)
builder.add_node("answer_reasoning", answer_reasoning)

builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", pick_tier)
builder.add_edge("answer_cheap", END)
builder.add_edge("answer_reasoning", END)

router_graph = builder.compile()
print("compiled:", list(router_graph.get_graph().nodes))

In [ ]:
# ============ VISUALISE ============
from IPython.display import Image, display

try:
    display(Image(router_graph.get_graph().draw_mermaid_png()))
except Exception as exc:  # rendering needs network; the ASCII fallback is enough
    print(f"(mermaid render unavailable: {type(exc).__name__})")
    print(router_graph.get_graph().draw_ascii())

## 4. A mixed workload

The workload has to be **mixed**, and honestly mixed. If it is all hard, routing cannot help
and you will conclude the pattern is worthless. If it is all easy, routing looks free and
you will conclude it is magic. Real traffic is lopsided — mostly easy, occasionally not —
so this set is too.

In [ ]:
# ============ WORKLOAD ============
# (question, expected-substring, is_actually_hard) — the third field is ground truth for
# scoring the router itself, not something the router sees.
WORKLOAD: list[tuple[str, str, bool]] = [
    ("Extract the order id from: 'Order #A-4417 shipped Tuesday.' Reply with the id only.",
     "A-4417", False),
    ("What is the capital of Portugal? One word.", "lisbon", False),
    ("Convert 2:45 pm to 24-hour time. HH:MM only.", "14:45", False),
    ("Reply with the word 'ok' and nothing else.", "ok", False),
    ("Services deploy one per day Mon-Fri. auth before billing. search the day immediately "
     "after billing. notify Monday. reporting not Friday. Which deploys Friday? One word.",
     "search", True),
    ("A cache holds 4 entries, LRU eviction. Accesses: A B C D A E B. Which key is evicted "
     "when E is inserted? One letter.", "c", True),
    ("Retries use exponential backoff 2s, 4s, 8s. Total seconds waiting across all three? "
     "Number only.", "14", True),
]

print(f"{len(WORKLOAD)} requests — "
      f"{sum(1 for *_, h in WORKLOAD if not h)} easy, {sum(1 for *_, h in WORKLOAD if h)} hard")

## 5. Run the router, then both baselines

Three passes over the same workload. Only by measuring all three can you say whether routing
paid — "the router was cheap" means nothing without "cheaper than what".

In [ ]:
# ============ THREE PASSES ============
def run_routed() -> list[dict]:
    out = []
    for question, expected, truly_hard in WORKLOAD:
        state = router_graph.invoke({"question": question})
        out.append({
            "expected": expected, "truly_hard": truly_hard,
            "tier": state["tier_used"], "routed_hard": state["difficulty"] == "hard",
            "correct": expected.lower() in state["answer"].lower(),
            "seconds": state["seconds"],
        })
    return out


def run_fixed(llm, tier: str) -> list[dict]:
    out = []
    for question, expected, truly_hard in WORKLOAD:
        started = time.perf_counter()
        reply = llm.invoke([HumanMessage(content=question)])
        text = (reply.content or "").strip()
        out.append({
            "expected": expected, "truly_hard": truly_hard,
            "tier": tier, "routed_hard": tier == "reasoning",
            "correct": expected.lower() in text.lower(),
            "seconds": time.perf_counter() - started,
        })
    return out


routed = run_routed()
all_cheap = run_fixed(cheap_llm, "cheap")
all_reasoning = run_fixed(reasoning_llm, "reasoning")
print("three passes complete")

In [ ]:
# ============ THE PAYOFF TABLE ============
def report(name: str, rows: list[dict]) -> None:
    acc = sum(r["correct"] for r in rows) / len(rows)
    secs = sum(r["seconds"] for r in rows)
    calls = f'{sum(1 for r in rows if r["tier"] == "reasoning")}/{len(rows)}'
    print(f"{name:<16}{acc:>9.0%}{secs:>11.1f}{calls:>18}")


print(f"{'strategy':<16}{'accuracy':>9}{'total sec':>11}{'reasoning calls':>18}")
print("-" * 54)
report("all cheap", all_cheap)
report("routed", routed)
report("all reasoning", all_reasoning)

print("\nRouting is worthwhile when its accuracy is at or near all-reasoning,")
print("while its reasoning-call count is far closer to all-cheap.")

### Discussion of the output

The shape you are looking for: **routed accuracy tracks all-reasoning, routed reasoning-call
count tracks all-cheap.** That gap is the saving, and it is the entire argument for the
pattern.

If routed accuracy sits below all-reasoning, the router is misclassifying hard questions as
easy — see the next section. If routed cost sits near all-reasoning, the router is calling
everything hard, and you are paying for triage on top of reasoning: strictly worse than
having no router.

## 6. The router is a model call, and it can be wrong

This is the part that separates a demo from a design. The triage step does not observe
difficulty, it *predicts* it — so it has a confusion matrix, and the two mistakes cost
wildly different amounts.

In [ ]:
# ============ ROUTER CONFUSION MATRIX ============
tp = sum(1 for r in routed if r["truly_hard"] and r["routed_hard"])
fn = sum(1 for r in routed if r["truly_hard"] and not r["routed_hard"])
fp = sum(1 for r in routed if not r["truly_hard"] and r["routed_hard"])
tn = sum(1 for r in routed if not r["truly_hard"] and not r["routed_hard"])

print(f"  hard, sent to reasoning (right)      : {tp}")
print(f"  easy, sent to cheap     (right)      : {tn}")
print(f"  easy, sent to reasoning (overspend)  : {fp}   <- costs money")
print(f"  hard, sent to cheap     (wrong answer): {fn}   <- costs correctness")

missed = [r for r in routed if r["truly_hard"] and not r["routed_hard"]]
if missed:
    print(f"\n{len(missed)} hard question(s) were routed cheap; "
          f"{sum(1 for r in missed if not r['correct'])} of those came back wrong.")

### Discussion: the two errors are not symmetric

- **Easy sent to reasoning** wastes money. Annoying, bounded, and visible on a bill.
- **Hard sent to cheap** returns a confidently wrong answer. Unbounded, and invisible until
  someone downstream acts on it.

So the router should be **biased toward over-escalating**. Tune the triage prompt so
borderline cases land in `hard`, and accept the overspend as insurance.

That asymmetry also fixes the fallback direction. The tempting design is *"try cheap, detect
a bad answer, retry on reasoning"* — but detecting a confidently wrong answer is exactly the
hard problem you do not have a solution for. **The workable direction is to escalate on
uncertainty before answering, not to demote after.** If you can reliably tell that an answer
is wrong, you did not need the router; you needed that detector.

## Key takeaways

1. **Routing by model tier is the Routing pattern on a different axis** — the conditional
   edge picks a model, not a handler. The graph shape is identical to `Routing.ipynb`.
2. **Measure against both baselines.** "The router was cheap" is meaningless without
   all-cheap and all-reasoning beside it.
3. **The router costs something.** Triage is a model call on every request, so a router that
   escalates everything is worse than no router at all.
4. **The two routing errors are not symmetric.** Over-escalation costs money; under-escalation
   costs correctness. Bias toward over-escalating.
5. **Escalate on uncertainty, do not demote after the fact.** A reliable wrong-answer detector
   would make the router unnecessary.
6. **Two constraints will bite you at call time, not construction time:** `reasoning_effort`
   is silently meaningless on a non-reasoning model, and o-series models reject a non-default
   temperature — which is why the cheap tier passes `temperature=0` and the reasoning tier
   passes `1`.

### Where this came from

- `01_Foundations/.../Reasoning_and_Model_Selection/01_Reasoning_vs_NonReasoning.ipynb`
- `01_Foundations/.../Reasoning_and_Model_Selection/02_Reasoning_Effort_Levers.ipynb`